In [1]:
import os

# Set working directory to project root always
# Works regardless of where the notebook is saved
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
os.chdir(project_root)

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\DELL\OneDrive\Documents\SRM\Churn_Analysis


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
df = pd.read_csv(
    'data/banking/Churn_Modelling.csv'
)

print("Shape:", df.shape)

df.head()

Shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.drop(
    columns=[
        'RowNumber',
        'CustomerId',
        'Surname'
    ],
    inplace=True
)

print(df.shape)

(10000, 11)


In [5]:
print(df.isnull().sum())

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [6]:
cat_cols = [
    'Geography',
    'Gender'
]

print(cat_cols)

['Geography', 'Gender']


In [7]:
le = LabelEncoder()

for col in cat_cols:

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    print(
        f"Encoded: {col}"
    )

Encoded: Geography
Encoded: Gender


In [8]:
print(
    df.select_dtypes(
        include='object'
    ).columns.tolist()
)

[]


In [9]:
import os

os.makedirs(
    'outputs',
    exist_ok=True
)

df.to_csv(
    'outputs/banking_clean.csv',
    index=False
)

print(
    "banking_clean.csv saved."
)

banking_clean.csv saved.


In [10]:
X = df.drop(
    columns=['Exited']
)

y = df['Exited']

print(
    "Features:",
    X.shape
)

print(
    "Target:",
    y.shape
)

Features: (10000, 10)
Target: (10000,)


In [11]:
scale_cols = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'EstimatedSalary'
]

print(scale_cols)

['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']


In [12]:
scaler = StandardScaler()

X[scale_cols] = scaler.fit_transform(
    X[scale_cols]
)

print(
    X.head()
)

   CreditScore  Geography  Gender       Age    Tenure   Balance  \
0    -0.326221          0       0  0.293517 -1.041760 -1.225848   
1    -0.440036          2       0  0.198164 -1.387538  0.117350   
2    -1.536794          0       0  0.293517  1.032908  1.333053   
3     0.501521          0       0  0.007457 -1.387538 -1.225848   
4     2.063884          2       0  0.388871 -1.041760  0.785728   

   NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  
0      -0.911583          1               1         0.021886  
1      -0.911583          0               1         0.216534  
2       2.527057          1               0         0.240687  
3       0.807737          0               0        -0.108918  
4      -0.911583          1               1        -0.365276  


In [13]:
import joblib

joblib.dump(
    scaler,
    'outputs/scaler_banking.pkl'
)

print(
    "Scaler saved."
)

Scaler saved.


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "\nTrain Churn Rate:",
    round(y_train.mean(),3)
)

print(
    "Test Churn Rate:",
    round(y_test.mean(),3)
)

X_train: (8000, 10)
X_test: (2000, 10)

Train Churn Rate: 0.204
Test Churn Rate: 0.204


In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_sm, y_train_sm = smote.fit_resample(
    X_train,
    y_train
)

print("SMOTE completed successfully")

SMOTE completed successfully


In [17]:
print("Before SMOTE")

print(
    y_train.value_counts()
)

print("\nAfter SMOTE")

print(
    pd.Series(
        y_train_sm
    ).value_counts()
)

Before SMOTE
Exited
0    6370
1    1630
Name: count, dtype: int64

After SMOTE
Exited
1    6370
0    6370
Name: count, dtype: int64


In [18]:
np.save(
    'outputs/X_train_banking.npy',
    X_train_sm
)

np.save(
    'outputs/X_test_banking.npy',
    X_test.values
)

np.save(
    'outputs/y_train_banking.npy',
    y_train_sm
)

np.save(
    'outputs/y_test_banking.npy',
    y_test.values
)

print(
    "Arrays saved successfully."
)

Arrays saved successfully.


In [19]:
pd.Series(
    X_train.columns
).to_csv(
    'outputs/banking_feature_names.csv',
    index=False
)

print(
    "Feature names saved."
)

Feature names saved.


In [20]:
print(
    "X_train_sm:",
    X_train_sm.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_train_sm:",
    len(y_train_sm)
)

print(
    "y_test:",
    len(y_test)
)

print("\nBalanced Classes:")

print(
    pd.Series(
        y_train_sm
    ).value_counts()
)

X_train_sm: (12740, 10)
X_test: (2000, 10)
y_train_sm: 12740
y_test: 2000

Balanced Classes:
Exited
1    6370
0    6370
Name: count, dtype: int64


In [21]:
print(
    pd.read_csv(
        'outputs/banking_feature_names.csv'
    ).iloc[:,0].tolist()
)

['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
